# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"Keywords: {metadata['keywords']}")
print(f"Data Collection Raw Data: {metadata['dataCollectionRawData']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List all record sets and their IDs
record_sets_info = []
for rs in dataset.record_sets:
    info = {
        'id': rs.id,
        'name': rs.name,
        'fields': [field.id for field in rs.fields]
    }
    record_sets_info.append(info)

print("Record Sets Summary:")
for info in record_sets_info:
    print(f"- RecordSet @id: {info['id']}, name: {info['name']}")
    print(f"  Fields @id's: {info['fields']}")

# Show a sample record from each record set
for rs_info in record_sets_info:
    print(f"\nSample record from {rs_info['name']} (@id: {rs_info['id']}):")
    sample = next(dataset.records(record_set=rs_info['id']), None)
    print(sample)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using @id
record_sets = [rs_info['id'] for rs_info in record_sets_info]
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"RecordSet {record_set} loaded. Columns: {df.columns.tolist()}")
    if not df.empty:
        print(df.head(1))

# As example, select the first record set for detailed preview
main_record_set = record_sets[0]
print(f"\nColumns in DataFrame for record set @id {main_record_set}:")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data. Reference fields via their `@id`.

In [ ]:
# Example EDA: Use clinical features (Table 1) for filtering and normalization
# Find a numeric field (@id) from Table 1 (e.g., age_at_diagnosis)

# Get the main clinical record set fields
clinical_fields = record_sets_info[0]['fields']
print(f"Fields in Main Record Set: {clinical_fields}")

# Assume 'age_at_diagnosis' field exists and is named as its @id. If not, use the real @id from the printed summary.
numeric_field = None
for col in dataframes[main_record_set].columns:
    if 'age' in col or 'age_at_diagnosis' in col:
        numeric_field = col
        break

if numeric_field:
    threshold = 30
    filtered_df = dataframes[main_record_set][dataframes[main_record_set][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field: pick 'infertility' or similar by @id
    group_field = None
    for col in dataframes[main_record_set].columns:
        if 'infertility' in col:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis in the main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization of age distribution
if numeric_field and not dataframes[main_record_set].empty:
    plt.figure(figsize=(6,4))
    dataframes[main_record_set][numeric_field].hist(bins=5)
    plt.title(f"Distribution of {numeric_field} in Clinical Features RecordSet")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Relationship between age and infertility status
    if group_field:
        plt.figure(figsize=(6,4))
        for label in dataframes[main_record_set][group_field].unique():
            subset = dataframes[main_record_set][dataframes[main_record_set][group_field]==label]
            plt.hist(subset[numeric_field], alpha=0.5, label=str(label))
        plt.title(f"{numeric_field} Distribution by {group_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.legend()
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and analyzed the FAIR^2 dataset using the `mlcroissant` library. 

- We accessed dataset metadata, clinical, laboratory, imaging, and variant tables by their Croissant `@id`s.
- Sampled records and extracted all fields dynamically for each record set.
- Applied filtering and normalization (e.g., age at diagnosis), and grouped by categorical attributes (e.g., infertility status).
- Visualized numeric distributions split by categorical features.

Due to the limited sample (N=3), deeper statistical analysis is restricted. The FAIR^2 dataset supports exploration and reproducible research in rare disorder genotype–phenotype correlations.